# Aprendizaje federado continuo: experimento CIFAR-100 en GPU

Ejecuta `cfl.py comparar` (buffers 0/5/10/20, 5 épocas locales) en sus tres variantes **a la vez**:
replay normal, lotes equilibrados y lotes equilibrados + destilación.

**Antes de ejecutar, en el panel derecho (*Session options*):**
1. **Accelerator → GPU T4 x2** (no P100: las versiones recientes de PyTorch ya no la soportan).
2. **Internet → On** (requiere tener el teléfono verificado en Kaggle).

**Para que siga aunque cierres el navegador:** *Save Version* → *Save & Run All (Commit)*.
Al terminar, los resultados están en la pestaña *Output* de esa versión (`resultados_kaggle.tar.gz`).

Una sesión de Kaggle dura como máximo 12 h. Por eso, por defecto se ejecuta **una semilla (42)**;
las semillas 43 y 44 se lanzan en otra sesión cambiando `SEEDS` abajo.

In [ ]:
# ---------------------------------------------------------------- parámetros
SEEDS = [42]                 # segunda sesión: [43, 44]
BUFFERS = [0, 5, 10, 20]
VARIANTES = ["normal", "balanced", "lwf"]
COMMIT = "8e77f12"           # revisión del código en GitHub (la misma que la imagen Docker)

In [ ]:
!nvidia-smi -L

In [ ]:
%%bash -s "$COMMIT"
set -e
cd /kaggle/working
rm -rf cfl-practicas
git clone -q https://github.com/Esteban7754/cfl-practicas.git
cd cfl-practicas && git checkout -q "$1" && git log -1 --oneline

## Entorno: mismas versiones fijadas que la imagen Docker, con PyTorch CUDA

Python 3.13 y `requirements.lock` sin el sufijo `+cpu` (igual que `TORCH_VARIANT=cuda` en el Dockerfile).
Si la versión de PyTorch de PyPI no funciona con el controlador de Kaggle, se prueba la **misma versión**
compilada para otras versiones de CUDA.

In [ ]:
%%bash
set -e
cd /kaggle/working/cfl-practicas
pip install -q uv
uv venv -q -p 3.13 /tmp/venv
sed -e '/^--extra-index-url/d' -e 's/+cpu//' requirements.lock > /tmp/req-cuda.txt
uv pip install -q -p /tmp/venv/bin/python -r /tmp/req-cuda.txt

ok() { /tmp/venv/bin/python -c "import torch; x = torch.ones(8, device='cuda'); assert float((x * 2).sum()) == 16" 2>/dev/null; }
if ! ok; then
    TORCH=$(grep '^torch==' /tmp/req-cuda.txt); VISION=$(grep '^torchvision==' /tmp/req-cuda.txt)
    for cu in cu130 cu128 cu126; do
        echo "PyTorch de PyPI no usa la GPU; probando $TORCH con $cu"
        uv pip install -q -p /tmp/venv/bin/python --reinstall "$TORCH" "$VISION" \
            --index-url "https://download.pytorch.org/whl/$cu" && ok && break || true
    done
fi
ok || { echo "[ERROR] PyTorch no puede usar la GPU. ¿Accelerator = GPU T4 x2?"; exit 1; }
/tmp/venv/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

In [ ]:
import os
os.environ.pop("PYTHONPATH", None)   # sin el sitecustomize de Kaggle
os.environ.update(
    HF_HOME="/tmp/hf",                      # caché fuera de /kaggle/working (no va al Output)
    MPLBACKEND="Agg",
    PYTHONHASHSEED="0",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",      # algoritmos deterministas en CUDA
)
REPO = "/kaggle/working/cfl-practicas"
PY = "/tmp/venv/bin/python"

# CIFAR-100 en la revisión fijada; después los experimentos trabajan sin red (sin descargas simultáneas).
!cd {REPO} && {PY} docker/descargar_datos.py
os.environ["CFL_DATA_PRELOADED"] = "1"
!cd {REPO} && {PY} cfl.py info

In [ ]:
!cd {REPO} && {PY} -m unittest -q 2>&1 | tail -n 4

## Experimento: las tres variantes en paralelo

Con lotes de 32 imágenes una sola variante no llena la GPU, así que se reparten entre las GPUs disponibles.
Cada variante es independiente y da el mismo resultado que si se ejecutara sola.

In [ ]:
import subprocess, time, re
from pathlib import Path

import torch
n_gpus = max(torch.cuda.device_count(), 1)
EXTRA = {
    "normal": [],
    "balanced": ["--replay-mix", "balanced"],
    "lwf": ["--replay-mix", "balanced", "--distill-weight", "1"],
}
procs, logs = {}, {}
for i, name in enumerate(VARIANTES):
    logs[name] = Path(f"/kaggle/working/{name}.log")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(i % n_gpus)}
    cmd = [PY, "cfl.py", "comparar", "--seeds", *map(str, SEEDS), "--buffers", *map(str, BUFFERS),
           "--threads", "1", *EXTRA[name]]
    print(f"[GPU {i % n_gpus}] {name}: {' '.join(cmd[1:])}")
    procs[name] = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=open(logs[name], "w"), stderr=subprocess.STDOUT)
    time.sleep(2)

start = time.time()
def last(name, pattern=r"Ronda|FASE|Semilla|ERROR|Traceback|terminada"):
    lines = [l for l in logs[name].read_text(errors="replace").splitlines() if re.search(pattern, l)]
    return lines[-1].strip() if lines else "(arrancando)"

while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    print(f"--- {(time.time() - start) / 3600:.1f} h")
    for name in procs:
        print(f"  {name:9s} {last(name)}")

for name, p in procs.items():
    print(f"{name}: {'OK' if p.returncode == 0 else f'FALLÓ (código {p.returncode}), ver {logs[name]}'}")

In [ ]:
%%bash
cd /kaggle/working
for f in cfl-practicas/resultados/*/; do
    echo "=================== $f"
    cat "$f"/resumen_semillas.log 2>/dev/null || grep -A12 "^=====" "$f"/ejecucion.log | tail -n 14
done
tar -czf resultados_kaggle.tar.gz --exclude='*.pt' -C cfl-practicas resultados -C /kaggle/working $(ls *.log)
ls -lh resultados_kaggle.tar.gz